# Fake News Detection System - Exploratory Data Analysis & Model Evaluation

This notebook contains the complete development pipeline for the **Fake News Detection System**. We walk through the following phases:
1. **Data Loading & Merging**
2. **Exploratory Data Analysis (EDA)**
3. **Text Cleaning & Preprocessing (NLP)**
4. **Feature Extraction (TF-IDF Vectorization)**
5. **Model Training (Logistic Regression, Naive Bayes, Random Forest, Passive Aggressive)**
6. **Model Evaluation & Comparison**
7. **Best Model Selection and Export**

In [ ]:
import os
import re
import string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import PassiveAggressiveClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Setting plotting configurations
sns.set_theme(style="whitegrid")
%matplotlib inline

## 1. Data Loading & Merging
We load `Fake.csv` and `True.csv` and add their respective labels ($0$ for Fake and $1$ for Real).

In [ ]:
# Ensure datasets are present. If not, generate them.
fake_path = os.path.join("..", "dataset", "Fake.csv")
true_path = os.path.join("..", "dataset", "True.csv")

if not os.path.exists(fake_path) or not os.path.exists(true_path):
    # Fallback path if running notebook inside the notebooks directory
    fake_path = os.path.join("dataset", "Fake.csv")
    true_path = os.path.join("dataset", "True.csv")
    
if not os.path.exists(fake_path) or not os.path.exists(true_path):
    print("Dataset files not found in normal search paths. Please execute generate_mock_data.py first!")
else:
    fake = pd.read_csv(fake_path)
    true = pd.read_csv(true_path)
    print(f"Fake News Shape: {fake.shape}")
    print(f"True News Shape: {true.shape}")

In [ ]:
# Add Labels
fake["label"] = 0
true["label"] = 1

# Combine datasets
data = pd.concat([fake, true], ignore_index=True)
print(f"Merged Dataset Shape: {data.shape}")
data.head()

## 2. Exploratory Data Analysis (EDA)
We look for duplicate entries, class balance, subject count distributions, and news text lengths.

In [ ]:
# Check class balance
plt.figure(figsize=(6, 4))
sns.countplot(data=data, x='label', hue='label', palette='viridis', legend=False)
plt.title('Distribution of Fake vs Real News')
plt.xticks([0, 1], ['Fake (0)', 'Real (1)'])
plt.show()

print(data['label'].value_counts())

In [ ]:
# Distribution of Subject categories
plt.figure(figsize=(10, 5))
sns.countplot(data=data, y='subject', hue='subject', palette='muted', order=data['subject'].value_counts().index, legend=False)
plt.title('Articles Count by Subject Category')
plt.show()

In [ ]:
# Remove Null and Duplicate Records
data.dropna(subset=['text', 'label'], inplace=True)
print(f"Duplicates count: {data.duplicated(subset=['text']).sum()}")
data.drop_duplicates(subset=['text'], inplace=True)
print(f"Cleaned dataset shape: {data.shape}")

## 3. Text Preprocessing & Cleaning
We define our `clean_text` function to lowercase the articles, remove URLs, HTML tags, punctuation, numbers, stopwords, and run Porter stemming.

In [ ]:
# Ensure NLTK packages downloaded
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def clean_text(text):
    if not isinstance(text, str):
        return ""
    
    # Lowercase
    text = text.lower()
    # Remove URLs
    text = re.sub(r'https?://\s*\S+|www\.\S+', '', text)
    # Remove HTML Tags
    text = re.sub(r'<.*?>', '', text)
    # Remove Punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Remove numbers
    text = re.sub(r'\w*\d\w*', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Tokenize and Stem
    words = word_tokenize(text)
    cleaned_words = [stemmer.stem(word) for word in words if word not in stop_words]
    
    return " ".join(cleaned_words)

In [ ]:
# Demonstration of cleaning
sample_text = "BREAKING NEWS: Shocking discovery! NASA scientists found water on Mars! Check out: http://nasa.gov/water-mars"
print("Original:", sample_text)
print("Cleaned :", clean_text(sample_text))

In [ ]:
# Preprocess whole dataset
print("Preprocessing text data... (This may take a moment)")
data['cleaned_text'] = data['text'].apply(clean_text)
data = data[data['cleaned_text'] != ""]
data.head()

## 4. Feature Extraction
We convert cleaned text strings into numerical vectors using a TF-IDF Vectorizer with unigrams and bigrams up to 5,000 features.

In [ ]:
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X = vectorizer.fit_transform(data['cleaned_text'])
y = data['label'].values
print(f"TF-IDF Matrix Shape: {X.shape}")

## 5. Model Training & Evaluation
We split the dataset into $80\%$ training and $20\%$ testing set, and train 4 different algorithms.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train features shape: {X_train.shape}")
print(f"Test features shape: {X_test.shape}")

In [ ]:
# Dictionary of classifiers
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Naive Bayes": MultinomialNB(),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Passive Aggressive": PassiveAggressiveClassifier(max_iter=1000, random_state=42)
}

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results[name] = acc
    
    print(f"\n=== {name} ===")
    print(f"Accuracy: {acc:.4%}")
    print(classification_report(y_test, y_pred))

## 6. Model Comparison

In [ ]:
# Visualizing model accuracies
plt.figure(figsize=(10, 6))
names = list(results.keys())
accuracies = list(results.values())
bars = sns.barplot(x=names, y=accuracies, hue=names, palette='viridis', legend=False)
plt.title('Comparison of Classifier Accuracies', fontsize=14)
plt.ylabel('Accuracy Score', fontsize=12)
plt.ylim(0, 1.1)

for bar in bars.patches:
    val = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, val + 0.01, f"{val*100:.2f}%", 
             ha='center', va='bottom', fontweight='bold')

plt.show()

## 7. Best Model Selection & Saving

In [ ]:
best_model_name = max(results, key=results.get)
best_acc = results[best_model_name]
print(f"Selecting best model: {best_model_name} ({best_acc:.2%})")

# Export models to models directory
os.makedirs("../models", exist_ok=True)
best_model_obj = models[best_model_name]

joblib.dump(best_model_obj, "../models/fake_news_model.pkl")
joblib.dump(vectorizer, "../models/tfidf_vectorizer.pkl")
print("Models exported successfully!")